# Use Data constructions and Boverket climate values

Data already contains Boverket Klimatdatabas records. This example explicitly loads the catalogue, assigns a 1960 multi-family construction set, and replaces windows. The outline is a teaching assumption, not an observed building.

In [ ]:
from epsm_data import Catalogue
from teaching_case import make_baseline, renovation_options, show_table

from epsm_lca import AssessmentSettings, assess_renovation

catalogue = Catalogue.load()
archetype = catalogue.for_year("multi_family", 1960, state="existing")
baseline = make_baseline(archetype.construction_set)
windows, _ = renovation_options(baseline)
renovated = baseline.renovate(windows)

Select conservative production values explicitly. The catalogue provides production (A1A3), transport (A4), and installation-waste (A5.1) values. **A5.1 is not the entire installation stage.** This first comparison deliberately covers only those available stages and excludes operating energy. Removal quantities are still retained; removal impacts are outside this particular selection.

In [ ]:
settings = AssessmentSettings.from_building(
    renovated,
    period_years=30,
    catalogue=catalogue,
    production="conservative",
    include_energy=False,
    material_modules=("A1A3", "A4", "A5.1"),
)
report = assess_renovation(renovated, settings)
assert report.complete
show_table(report)
print("Removed windows:", report.quantities(action="removed", by_material=True))
print("Factors and source references:", settings.to_dict()["material_factors"])

Boverket window resource 6000000104 provides 2.5 kg CO₂e/kg for conservative production and an explicit **38.2 kg/m²** conversion. Production is therefore **95.5 kg CO₂e/m²**. Window area is known; the package does not invent a measured window mass. Source values, conversion, version 02.07.000 and the GWP-GHG AR4 measure remain in the factor references.

In [ ]:
import math

window_rows = [r for r in renovated.quantities() if r.kind == "window"]
factor = settings.material_factors[window_rows[0].material_id]["A1A3"]
assert factor.unit == "m2" and math.isclose(factor.value, 95.5)
assert all(r.mass_kg is None for r in window_rows)
assert "02.07.000" in factor.source

For a wider renovation assessment, keep the default selection and supply sourced removal and energy factors. If they are missing, the report tells you. The Boverket helper does not guess them or silently use installation waste as complete installation impact.

In [ ]:
wider = AssessmentSettings.from_building(
    renovated, period_years=30, catalogue=catalogue, include_energy=False
)
incomplete = assess_renovation(renovated, wider)
assert not incomplete.complete
assert any("C4" in r for r in incomplete.stages[0].requirements)
print(incomplete.stages[0].requirements)